# AI4SAW — Notebook 02: Extraction Demo

Demonstrates the three extraction tasks:
1. **NER** — named entity recognition via few-shot prompting
2. **Relation extraction** — subject–predicate–object triples via chain-of-thought
3. **Event classification** — zero-shot first, few-shot fallback

All results are Pydantic-validated and ready for export.

In [ ]:
import sys
sys.path.insert(0, '..')

from ai4saw.core.config import settings
print(f'Provider: {settings.provider} | Model: {settings.default_model}')

## Sample text

Replace this with a real chunk from your corpus.

In [ ]:
SAMPLE_TEXT = """
Witness testimony collected by the ICTY confirmed that Ratko Mladic ordered
the execution of prisoners held at the Kravica warehouse near Bratunac in
July 1995. Survivors reported that Bosniak men were transported by Drina Corps
soldiers and forced to dig their own graves before being shot. The Human Rights
Watch documented at least 1,000 victims at this location alone. The UN Security
Council Resolution 827 established the tribunal empowered to prosecute such
crimes against humanity.
""".strip()

CHUNK_ID = 'demo_chunk_001'
print(SAMPLE_TEXT)

## 1. Named Entity Recognition

In [ ]:
from ai4saw.extraction.ner import extract_entities

ner_result = extract_entities(SAMPLE_TEXT, CHUNK_ID)

print(f'Entities extracted: {len(ner_result.entities)}\n')
for e in ner_result.entities:
    print(f'  [{e.label:20s}] {e.text!r:40s}  conf={e.confidence:.2f}')

## 2. Relation Extraction

In [ ]:
from ai4saw.extraction.relations import extract_relations

rel_result = extract_relations(SAMPLE_TEXT, CHUNK_ID)

print(f'Relations extracted: {len(rel_result.relations)}\n')
for r in rel_result.relations:
    print(f'  {r.subject!r:30s}  --[{r.predicate}]-->  {r.object!r}')
    if r.location:
        print(f'    @ {r.location}')
    if r.date:
        print(f'    date: {r.date}')
    print(f'    conf={r.confidence:.2f}  evidence: {r.evidence[:80]!r}')
    print()

## 3. Event Classification

In [ ]:
from ai4saw.extraction.events import classify_event

event_result = classify_event(SAMPLE_TEXT, CHUNK_ID)

print(f'Event type:   {event_result.event_type.value}')
print(f'Confidence:   {event_result.confidence:.2f}')
print(f'Location:     {event_result.location}')
print(f'Date:         {event_result.date}')
print(f'Perpetrator:  {event_result.perpetrator}')
print(f'Victim group: {event_result.victim_group}')

## 4. Batch extraction from ChromaDB

Runs all three extractors over every chunk currently in the vector store.

In [ ]:
from ai4saw.ingestion.embedder import get_vector_store
from ai4saw.extraction.ner import extract_entities_batch
from ai4saw.extraction.relations import extract_relations_batch
from ai4saw.extraction.events import classify_events_batch

store = get_vector_store()
collection = store._collection
raw = collection.get(include=['documents', 'ids'])

texts = raw.get('documents') or []
ids = raw.get('ids') or []
pairs = list(zip(texts, ids))

if not pairs:
    print('No chunks in ChromaDB — run notebook 01 first.')
else:
    print(f'Running extraction on {len(pairs)} chunks...')
    ner_results = extract_entities_batch(pairs[:5])   # limit for demo
    print(f'NER done: {sum(len(r.entities) for r in ner_results)} entities total')

## 5. Zero-shot vs few-shot comparison

Demonstrates the publishable finding: direct comparison of zero-shot vs few-shot
event classification on the same chunks.

In [ ]:
import yaml
from pathlib import Path
from langchain_core.messages import HumanMessage, SystemMessage
from ai4saw.core.providers import get_llm
from ai4saw.core.config import settings

zero_shot_path = settings.prompts_dir / 'events_zero_shot.yaml'
prompt = yaml.safe_load(zero_shot_path.read_text())

llm = get_llm()
user_content = prompt['template'].replace('{chunk_text}', SAMPLE_TEXT)
messages = [
    SystemMessage(content=prompt['system']),
    HumanMessage(content=user_content),
]

response = llm.invoke(messages)
print('Zero-shot raw output:')
print(response.content)